In [1]:
import os
from pathlib import Path
import math
import pytest
import pandas as pd
import torch
from torch import nn
from torch_geometric.loader import DataLoader

In [2]:
import sys
sys.path.append(str(Path().resolve().parents[1]))
sys.path.append(str(Path().resolve().parents[0]))

In [3]:
from SynRxNet.src.datasets import create_dataset, Splitter, FeatureEngineer

In [4]:
DATA_ROOT = Path("../data")
CLEANED_PATH = DATA_ROOT / "processed" / "cleaned_drugcomb.csv"
CCLE_PCA_PATH = DATA_ROOT / "processed" / "ccle_cell_line_pca_100.csv"
print(DATA_ROOT,Path.exists(CLEANED_PATH), Path.exists(CCLE_PCA_PATH))

../data True True


## Basic Sanity Checks

In [ ]:
@pytest.mark.skipif(not CLEANED_PATH.exists(), reason="Cleaned data file not found")
def test_cleaned_dataset_structure():
    df = pd.read_csv(CLEANED_PATH)
    required = {"ID", "Drug1", "Drug2", "Cell line", "ZIP", "smiles_drug1", "smiles_drug2"}
    missing = required - set(df.columns)
    assert not missing, f"Missing columns in cleaned dataset: {missing}"
    assert len(df) > 0, "Cleaned dataset is empty"
    print("Cleaned dataset structure is valid.")

In [ ]:
test_cleaned_dataset_structure()

## Splitter Sanity

In [ ]:
@pytest.mark.skipif(not CLEANED_PATH.exists(), reason="Cleaned data file not found")
@pytest.mark.parametrize("strategy", ["random", "lodo", "loco"])
def test_splitter_strategies(strategy):
    df = pd.read_csv(CLEANED_PATH)
    df = df.sample(min(200, len(df)), random_state=42).reset_index(drop=True)

    splitter = Splitter(strategy=strategy, seed=123)
    wut = splitter.split(df)
    train, val, test = wut

    # Basic Checks
    assert len(train) + len(val) + len(test) == len(df), "Data split sizes do not add up"
    assert len(set(train['ID']) & set(val['ID'])) == 0, "Train and Val sets overlap"
    assert len(set(train['ID']) & set(test['ID'])) == 0, "Train and Test sets overlap"
    assert len(set(val['ID']) & set(test['ID'])) == 0, "Val and Test sets overlap"
    print(f"{strategy} split: Train={len(train)}, Val={len(val)}, Test={len(test)}")

    if strategy == "lodo":
        # TODO: Implement LODO specific checks
        pass
    if strategy == "loco":
        train_cell_lines = set(train['Cell line'])
        test_cell_lines = set(test['Cell line'])
        assert train_cell_lines.isdisjoint(test_cell_lines), "Train and Test cell lines overlap in LOCO split"
    print(f"{strategy} splitter passed all tests.")

In [ ]:
test_splitter_strategies("random")
test_splitter_strategies("loco")
# test_splitter_strategies("lodo")

In [ ]:
test_splitter_strategies("lodo")

## FeatureEngineer Basic Operations

In [ ]:
def test_feature_engineer_smiles_and_descriptors():
    fe = FeatureEngineer(cache_dir=DATA_ROOT / "cache_test")
    smiles_valid = "CCO"
    smiles_invalid = "not_a_smiles"

    assert fe.validate_smiles(smiles_valid) is True
    assert fe.validate_smiles(smiles_invalid) is False

    emb = fe.encode_smiles(smiles_valid)
    assert isinstance(emb, torch.Tensor)
    assert emb.ndim == 1 and emb.shape[0] == 768

    desc = fe.compute_descriptors(smiles_valid)
    assert isinstance(desc, torch.Tensor)
    assert desc.ndim == 1
    assert desc.shape[0] == len(fe.desc_names)

    # 3D features
    coords = fe.compute_3d_features(smiles_valid)
    assert isinstance(coords, torch.Tensor)
    assert coords.ndim == 1  # flattened distance vector
    print(coords.shape)
    assert coords.shape[0] == 1000
    print("Feature engineering tests passed.")

In [ ]:
test_feature_engineer_smiles_and_descriptors()

## Dataset + PyG Dataloader integration

In [5]:
def _make_small_dataset(dataset_type = "graph", subset = "trian", strategy="random"):

    kwargs = {'missing_cell_policy': 'drop'}
    if CCLE_PCA_PATH.exists():
        kwargs['cell_line_features_path'] = str(CCLE_PCA_PATH)
    
    dataset = create_dataset(
        csv_path=str(CLEANED_PATH),
        dataset_type=dataset_type,
        split_strategy=strategy,
        split_seed=42,
        cache_dir=str(DATA_ROOT / "cache_test"),
        **kwargs
    )
    return dataset

In [6]:
@pytest.mark.skipif(not CLEANED_PATH.exists(), reason="Cleaned data file not found")
def test_graph_dataset_and_loader_basic():
    ds = _make_small_dataset(dataset_type="graph", subset="train", strategy="random")
    assert len(ds) > 0, "Graph dataset is empty"

    item = ds[0]
    print(item)

    # Keys must match what __getitem__ returns
    for k in ["graph1", "graph2", "cell_line", "label", "metadata"]:
        assert k in item

    g1 = item["graph1"]
    g2 = item["graph2"]
    assert hasattr(g1, "x") and hasattr(g1, "edge_index"), "Drug1 graph missing attributes"
    assert hasattr(g2, "x") and hasattr(g2, "edge_index"), "Drug2 graph missing attributes"

    loader = DataLoader(ds, batch_size=8, shuffle=True)
    batch = next(iter(loader))
    print(batch)

    # batch is a dict, so use dict access:
    assert "graph1" in batch and "graph2" in batch, "Batch missing graph keys"
    g1_batch = batch["graph1"]
    g2_batch = batch["graph2"]

    # These are DataBatch objects
    assert hasattr(g1_batch, "num_graphs") and hasattr(g2_batch, "num_graphs"), "Batched graphs missing num_graphs"
    assert g1_batch.num_graphs == 8, "Batch size mismatch for graph1"
    assert g2_batch.num_graphs == 8, "Batch size mismatch for graph2"
    print("Graph dataset and DataLoader tests passed.")


In [7]:
test_graph_dataset_and_loader_basic()

81 cell lines in synergy data could not be matched to PCA features: ['3D7', '786-0', 'A-375', 'A-673', 'BT-549', 'CAKI-1', 'CCRF-CEM', 'COLO 205', 'COLO 858', 'COLO320DM', 'CTR', 'DD2', 'DIPG25', 'DLD1', 'DU-145', 'EFM192B', 'EW-8', 'HB3', 'HCC-2998', 'HCT-15', 'HDLM-2', 'HL-60(TB)', 'HOP-62', 'HOP-92', 'HS 578T', 'Huh-7', 'JHH-136', 'K-562', 'KB-3-1', 'KB-ChR-8-5-11', 'KBM-7', 'L-1236', 'L-428', 'LOX IMVI', 'M14', 'MALME-3M', 'MDA-MB-231', 'MDA-MB-435', 'MDA-MB-468', 'MMAC-SF', 'MOLT-4', 'MZ7-mel', 'Mak', 'NCI-H226', 'NCI-H322M', 'NCI-H460', 'NCI-H522', 'NCI\\\\/ADR-RES', 'OCUBM', 'OVCAR-4', 'OVCAR-5', 'OVCAR-8', 'PA1', 'PC-3', 'RPMI-8226', 'RXF 393', 'Rh36', 'SF-268', 'SF-295', 'SF-539', 'SK-MEL-2', 'SK-MEL-28', 'SK-MEL-5', 'SK-OV-3', 'SMS-CTR', 'SN12C', 'SNB-19', 'SNB-75', 'SU-DIPG-XIII', 'SW-620', 'T-47D', 'TC-32', 'TC-71', 'TK-10', 'TMD8', 'U-HO1', 'UACC-257', 'UO-31', 'UWB1289', 'UWB1289+BRCA1', 'WM-115']
Dropping 44452 rows due to missing cell-line features (after mapping).


{'graph1': Data(x=[24, 5], edge_index=[2, 54], edge_attr=[54, 4]), 'graph2': Data(x=[29, 5], edge_index=[2, 66], edge_attr=[66, 4]), 'cell_line': tensor([ 2.7117e+01,  2.7477e+01,  5.0505e+01, -1.5724e+01,  7.3606e+01,
         1.2984e+01, -4.2328e+01,  1.2079e+01, -3.3369e+01,  1.8597e+01,
         1.2131e+02,  1.2750e+02, -4.8797e+01,  4.2852e+01,  1.3309e+00,
         2.2079e+01, -1.5031e+01,  4.3254e+01, -6.3720e+00,  7.7020e+00,
         1.9339e+00, -1.4637e+01, -5.5095e+00, -5.6980e+00, -1.0585e+01,
        -6.4885e+00, -7.5289e+00, -7.1240e+00, -7.8345e+00, -2.1730e+01,
         3.1810e+00,  2.1878e+00, -1.1858e+00, -7.7426e+00, -5.5422e+00,
        -1.6702e+00, -1.7611e+00,  7.0925e-01, -6.2099e+00, -1.4706e+00,
        -2.3275e+00, -6.8801e-13]), 'label': tensor([-6.6600, -0.9200,  1.8900,  2.0700]), 'metadata': {'ID': 56471, 'Drug1': 'MK-4827', 'Drug2': 'SN-38', 'cell_line': 'OVCAR3', 'ZIP': -6.66, 'Bliss': -0.92, 'Loewe': 1.89, 'HSA': 2.07, 'smiles1': 'C1CC(CNC1)C2=CC=C(C=C2

## Final pipeline test

In [12]:
import os
from pathlib import Path
import torch
from torch import nn
from torch_geometric.loader import DataLoader as PyGDataLoader

from src.datasets.splitter import Splitter
from src.datasets.feature_engineer import FeatureEngineer
from src.datasets.base import BaseDataset  # your final BaseDataset
from src.datasets.graph_dataset import GraphDataset  # assumes this wraps BaseDataset


DATA_ROOT = Path("../data")
CLEANED_PATH = DATA_ROOT / "processed" / "cleaned_drugcomb.csv"
CCLE_PCA_PATH = DATA_ROOT / "processed" / "ccle_cell_line_pca_100.csv"


class TinyGCN(nn.Module):
    def __init__(self, in_channels: int, cell_dim: int, hidden: int = 64):
        super().__init__()
        from torch_geometric.nn import GCNConv, global_mean_pool

        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.pool = global_mean_pool

        self.fc = nn.Sequential(
            nn.Linear(hidden * 2 + cell_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),  # predict ZIP only
        )

    def forward(self, batch):
        g1 = batch["graph1"]
        g2 = batch["graph2"]
        cell = batch["cell_line"]  # [B, cell_dim]

        # Drug 1
        x1 = torch.relu(self.conv1(g1.x, g1.edge_index))
        x1 = torch.relu(self.conv2(x1, g1.edge_index))
        pooled1 = self.pool(x1, g1.batch)  # [B, hidden]

        # Drug 2
        x2 = torch.relu(self.conv1(g2.x, g2.edge_index))
        x2 = torch.relu(self.conv2(x2, g2.edge_index))
        pooled2 = self.pool(x2, g2.batch)

        h = torch.cat([pooled1, pooled2, cell], dim=1)
        out = self.fc(h).squeeze(-1)  # [B]
        return out


def build_graph_dataset(subset: str = "train"):
    splitter = Splitter(strategy="random", seed=42)
    fe = FeatureEngineer(cache_dir="data/cache_test")

    ds = GraphDataset(
        csv_path=str(CLEANED_PATH),
        splitter=splitter,
        feature_engineer=fe,
        subset=subset,
        cell_line_features_path=str(CCLE_PCA_PATH),
        n_cell_line_components=None,          # PCA already done
        missing_cell_policy="drop",          # strict CCLE-only
    )
    return ds


def main():
    if not CLEANED_PATH.exists():
        raise FileNotFoundError(f"Missing {CLEANED_PATH}")
    if not CCLE_PCA_PATH.exists():
        raise FileNotFoundError(f"Missing {CCLE_PCA_PATH}")

    # Build datasets
    train_ds = build_graph_dataset("train")
    val_ds = build_graph_dataset("val")

    print(f"Train size after CCLE filtering: {len(train_ds)}")
    print(f"Val size after CCLE filtering:   {len(val_ds)}")

    # If it's too small, you'll see it here.
    if len(train_ds) < 50:
        print("Warning: very small train set; convergence may be noisy.")

    # Make loaders (small batches for debugging)
    train_loader = PyGDataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = PyGDataLoader(val_ds, batch_size=64, shuffle=False)

    # Inspect one batch to get dims
    first_batch = next(iter(train_loader))
    g1 = first_batch["graph1"]
    cell_feat = first_batch["cell_line"]
    in_channels = g1.x.size(-1)
    cell_dim = cell_feat.size(-1)
    print(f"Atom feature dim: {in_channels}, cell-line dim: {cell_dim}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TinyGCN(in_channels=in_channels, cell_dim=cell_dim, hidden=64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    def run_epoch(loader, train: bool):
        if train:
            model.train()
        else:
            model.eval()
        total_loss = 0.0
        n = 0
        with torch.set_grad_enabled(train):
            for batch in loader:
                # Move tensors to device
                batch["graph1"] = batch["graph1"].to(device)
                batch["graph2"] = batch["graph2"].to(device)
                batch["cell_line"] = batch["cell_line"].to(device)
                batch["label"] = batch["label"].to(device)

                preds = model(batch)
                zip_target = batch["label"][:, 0]  # ZIP only
                loss = criterion(preds, zip_target)

                if train:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                total_loss += loss.item() * zip_target.size(0)
                n += zip_target.size(0)
        return total_loss / max(1, n)

    # Train for a few epochs
    num_epochs = 5
    train_losses = []
    val_losses = []

    for epoch in range(1, num_epochs + 1):
        train_loss = run_epoch(train_loader, train=True)
        val_loss = run_epoch(val_loader, train=False)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    # Simple convergence check
    if len(train_losses) >= 2:
        improvement = train_losses[0] - train_losses[-1]
        print(f"Train loss improvement: {improvement:.4f}")
        if improvement <= 0:
            print("Warning: train loss did not improve; check features/splits.")
    print("Done.")


if __name__ == "__main__":
    main()


81 cell lines in synergy data could not be matched to PCA features: ['3D7', '786-0', 'A-375', 'A-673', 'BT-549', 'CAKI-1', 'CCRF-CEM', 'COLO 205', 'COLO 858', 'COLO320DM', 'CTR', 'DD2', 'DIPG25', 'DLD1', 'DU-145', 'EFM192B', 'EW-8', 'HB3', 'HCC-2998', 'HCT-15', 'HDLM-2', 'HL-60(TB)', 'HOP-62', 'HOP-92', 'HS 578T', 'Huh-7', 'JHH-136', 'K-562', 'KB-3-1', 'KB-ChR-8-5-11', 'KBM-7', 'L-1236', 'L-428', 'LOX IMVI', 'M14', 'MALME-3M', 'MDA-MB-231', 'MDA-MB-435', 'MDA-MB-468', 'MMAC-SF', 'MOLT-4', 'MZ7-mel', 'Mak', 'NCI-H226', 'NCI-H322M', 'NCI-H460', 'NCI-H522', 'NCI\\\\/ADR-RES', 'OCUBM', 'OVCAR-4', 'OVCAR-5', 'OVCAR-8', 'PA1', 'PC-3', 'RPMI-8226', 'RXF 393', 'Rh36', 'SF-268', 'SF-295', 'SF-539', 'SK-MEL-2', 'SK-MEL-28', 'SK-MEL-5', 'SK-OV-3', 'SMS-CTR', 'SN12C', 'SNB-19', 'SNB-75', 'SU-DIPG-XIII', 'SW-620', 'T-47D', 'TC-32', 'TC-71', 'TK-10', 'TMD8', 'U-HO1', 'UACC-257', 'UO-31', 'UWB1289', 'UWB1289+BRCA1', 'WM-115']
Dropping 44452 rows due to missing cell-line features (after mapping).
81 c

Train size after CCLE filtering: 10772
Val size after CCLE filtering:   1347
Atom feature dim: 5, cell-line dim: 42
Epoch 1: train_loss=48.6917, val_loss=46.6597
Epoch 2: train_loss=47.1010, val_loss=47.1084
Epoch 3: train_loss=46.5938, val_loss=45.4433
Epoch 4: train_loss=45.9221, val_loss=45.2094
Epoch 5: train_loss=45.3811, val_loss=43.7137
Train loss improvement: 3.3106
Done.
